# LeetCode 1020 — Number of Enclaves bằng Python

Link đề: https://leetcode.com/problems/number-of-enclaves/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Depth-First Search (DFS), Breadth-First Search (BFS), Union Find (hợp nhất tập hợp), Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def numEnclaves(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: cho lưới nhị phân `m x n`, trong đó `0` là ô biển và `1` là ô đất. Một **bước đi** là đi từ một ô đất sang ô đất **kề nhau theo 4 hướng**, hoặc **đi ra khỏi biên** lưới. Đếm số ô đất **không thể đi ra khỏi biên** dù đi bao nhiêu bước. Ý then chốt: thay vì kiểm tra từng ô đất có ra được biên không, hãy **đi ngược từ biên**: mọi ô đất nằm trên biên hoặc **liên thông** với một ô đất trên biên đều thoát được; các ô đất còn lại chính là "ô bị bao vây" (enclave). Vậy ta chạy **loang (flood fill) nhiều nguồn** từ mọi ô đất ở biên rồi **đếm** ô đất chưa bị loang tới.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ô đất "thoát được" nếu có đường đi qua các ô đất kề nhau tới một ô đất trên biên; đếm các ô đất không thoát được.
2. Nắm các công cụ Python cần thiết: ma trận dạng danh sách hai chiều, bốn hướng `((1,0),(-1,0),(0,1),(0,-1))`, `collections.deque` cho BFS, ngăn xếp bằng `list` cho DFS lặp, và cấu trúc **Union-Find** (DSU).
3. Hiểu nguyên lý: **đảo ngược bài toán**: thay vì từ mỗi ô đất tìm đường ra biên (tốn kém), đi từ **mọi ô đất ở biên** loang vào trong; ô đất bị loang là "thoát được".
4. Đi qua 4 cách: từ mỗi ô đất chạy BFS tìm đường ra biên (brute force, oracle), DFS lặp nhiều nguồn từ biên, BFS nhiều nguồn từ biên (cách nộp chính) và Union-Find với một nút ảo "bên ngoài".
5. Tránh các lỗi kinh điển: chỉ loang từ **một phần** của biên (quên hàng/cột cuối), chỉ loại các ô **nằm trên biên** mà không loang vào trong, dùng DFS **đệ quy** (tràn giới hạn đệ quy với lưới `500 x 500`), cho phép đi chéo (8 hướng), và đếm **số vùng** thay vì **số ô**.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `m x n` tên `grid`, trong đó `0` là **ô biển** và `1` là **ô đất**.

Một **bước đi** là đi từ một ô đất sang một ô đất khác **kề nhau theo 4 hướng** (trên, dưới, trái, phải), hoặc **đi ra khỏi biên** của `grid`.

Trả về **số ô đất** trong `grid` mà ta **không thể đi ra khỏi biên** của lưới bằng bất kỳ số bước đi nào.

**Ví dụ 1:** `grid = [[0,0,0,0],[1,0,1,0],[0,1,1,0],[0,0,0,0]]` -> `3`. Có ba ô `1` bị bao quanh bởi `0`, và một ô `1` không bị bao vây vì nó nằm trên biên (hàng 1, cột 0).

**Ví dụ 2:** `grid = [[0,1,1,0],[0,0,1,0],[0,0,1,0],[0,0,0,0]]` -> `0`. Mọi ô `1` hoặc nằm trên biên hoặc đi được tới biên.

**Ràng buộc:**

- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 500`
- `grid[i][j]` là `0` hoặc `1`.

In [1]:
from typing import List, Optional
from collections import deque
import copy
import random, time

DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))

examples = [
    ([[0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]], 3),
    ([[0, 1, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 0]], 0),
]
for grid, expected in examples:
    print(f"grid={grid} -> kỳ vọng {expected}")

grid=[[0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]] -> kỳ vọng 3
grid=[[0, 1, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 0]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Ma trận, ô trên biên và bốn hướng kề

Ô `(i, j)` nằm **trên biên** khi `i == 0` hoặc `i == m - 1` hoặc `j == 0` hoặc `j == n - 1`. Từ ô `(i, j)` đi tới bốn ô kề `(i + di, j + dj)` với `(di, dj)` thuộc `DIRS`; ô kề phải nằm trong lưới (`0 <= i + di < m` và `0 <= j + dj < n`) mới là ô thật.

In [2]:
grid = examples[0][0]
m, n = len(grid), len(grid[0])
def on_border(i, j): return i in (0, m - 1) or j in (0, n - 1)
print("ô trên biên có đất:", [(i, j) for i in range(m) for j in range(n) if grid[i][j] == 1 and on_border(i, j)])
print("ô đất bên trong:", [(i, j) for i in range(m) for j in range(n) if grid[i][j] == 1 and not on_border(i, j)])

ô trên biên có đất: [(1, 0)]
ô đất bên trong: [(1, 2), (2, 1), (2, 2)]


### 3.2. Loang (flood fill): DFS và BFS

Từ một ô đất xuất phát, **loang** là thăm mọi ô đất liên thông với nó (đi qua các ô đất kề nhau). DFS dùng ngăn xếp (hoặc đệ quy), BFS dùng hàng đợi; cả hai thăm mỗi ô **một lần** nên `O(số ô)`. Đánh dấu ô đã thăm để không thăm lại.

### 3.3. Loang nhiều nguồn (multi-source)

Bắt đầu loang đồng thời từ **nhiều** ô xuất phát: đưa tất cả vào hàng đợi (hoặc ngăn xếp) ngay từ đầu rồi chạy bình thường. Kết quả là tập mọi ô liên thông với **ít nhất một** nguồn. Ở đây nguồn là mọi ô đất trên biên: tập loang được chính là tập ô đất thoát được.

In [3]:
def reachable_from_border(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    dq = deque()
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1 and (i in (0, m - 1) or j in (0, n - 1)):
                seen[i][j] = True
                dq.append((i, j))
    while dq:
        i, j = dq.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                seen[a][b] = True
                dq.append((a, b))
    return seen

seen = reachable_from_border(examples[0][0])
for row, srow in zip(examples[0][0], seen):
    print(" ".join(("#" if s else "1") if v else "." for v, s in zip(row, srow)))
print("('#' = thoát được; '1' = bị bao vây; '.' = biển)")

. . . .
# . 1 .
. 1 1 .
. . . .
('#' = thoát được; '1' = bị bao vây; '.' = biển)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: lưới `4 x 4`

```
0 0 0 0
1 0 1 0
0 1 1 0
0 0 0 0
```

- Ô đất trên biên: `(1, 0)` (cột `0`). Loang từ nó: các ô kề `(0, 0)`, `(2, 0)`, `(1, 1)` đều là biển ⇒ không loang tới ô nào khác.
- Ô đất còn lại: `(1, 2)`, `(2, 1)`, `(2, 2)`. Chúng không được loang tới ⇒ bị bao vây: `3` ô.

Ví dụ 2: đất ở `(0, 1), (0, 2), (1, 2), (2, 2)`; ô `(0, 1)` và `(0, 2)` nằm trên biên; loang qua `(1, 2)` rồi `(2, 2)`: mọi ô đất đều thoát được ⇒ `0`.

## 5. Cách 1 — Brute Force: từ mỗi ô đất tìm đường ra biên

**Ý tưởng.** Với mỗi ô đất `(i, j)`, chạy một BFS **riêng** qua các ô đất kề nhau để xem có chạm tới một ô đất trên biên không (đi ra khỏi biên nghĩa là đang đứng ở ô trên biên). Đếm các ô đất mà BFS không tới được biên. Đúng theo định nghĩa của đề (không dùng ý tưởng đảo ngược) nên là **oracle độc lập**.

In [4]:
def enclaves_brute(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    count = 0
    for si in range(m):
        for sj in range(n):
            if grid[si][sj] != 1:
                continue
            seen = {(si, sj)}
            dq = deque([(si, sj)])
            escaped = False
            while dq and not escaped:
                i, j = dq.popleft()
                if i in (0, m - 1) or j in (0, n - 1):      # đang ở ô đất trên biên: có thể bước ra ngoài
                    escaped = True
                    break
                for di, dj in DIRS:
                    a, b = i + di, j + dj
                    if grid[a][b] == 1 and (a, b) not in seen:
                        seen.add((a, b))
                        dq.append((a, b))
            if not escaped:
                count += 1
    return count

for grid, expected in examples:
    got = enclaves_brute(grid)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = 3 OK
brute = 0 OK


**Độ phức tạp và nút thắt.** Mỗi ô đất chạy một BFS `O(m * n)`: tổng `O((m * n)^2)`; với lưới `500 x 500` là `~ 6 * 10^10`: không thể. Đo thử:

In [5]:
def random_grid(rng, m, n, p):
    return [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]

rng = random.Random(1020)
for size in (10, 20, 30, 40):
    grid = random_grid(rng, size, size, 0.7)
    t0 = time.perf_counter()
    enclaves_brute(grid)
    dt = time.perf_counter() - t0
    print(f"{size:2}x{size:<2}: {dt * 1000:9.1f} ms -> ngoại suy 500x500: ~{dt * (500 / size) ** 4 / 3600:9.1f} giờ")

10x10:       0.5 ms -> ngoại suy 500x500: ~      0.8 giờ
20x20:       7.2 ms -> ngoại suy 500x500: ~      0.8 giờ
30x30:      25.0 ms -> ngoại suy 500x500: ~      0.5 giờ
40x40:      62.5 ms -> ngoại suy 500x500: ~      0.4 giờ


Nút thắt: ta lặp lại việc "đi ra biên" cho từng ô; nhưng nhiều ô thuộc cùng một vùng đất liên thông nên có cùng đáp án. Ta nên **làm ngược**: xuất phát từ biên.

## 6. Ý tưởng thuật toán / bước đột phá

**Ô đất thoát được ⇔ liên thông (qua các ô đất) với một ô đất trên biên.**

Chứng minh ngắn: nếu ô `u` có đường đi (qua các ô đất kề nhau) tới một ô đất `w` trên biên thì từ `w` ta bước ra khỏi lưới nên `u` thoát được. Ngược lại nếu `u` thoát được thì cuối đường đi ta đứng ở một ô đất trên biên (bước ra khỏi lưới chỉ xảy ra từ ô trên biên), nên `u` liên thông với một ô đất trên biên.

Vậy: **loang từ mọi ô đất trên biên** (đa nguồn), đánh dấu mọi ô đất tới được; kết quả = (số ô đất) - (số ô đất đã đánh dấu).

| Cách | Ý tưởng |
|---|---|
| 2 | DFS lặp (ngăn xếp) đa nguồn từ biên |
| 3 | BFS đa nguồn từ biên (nộp) |
| 4 | Union-Find với nút ảo "bên ngoài": hợp nhất mọi ô đất biên với nút này |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS lặp đa nguồn từ biên

**Giải thích từng đoạn.**

- Duyệt mọi ô trên biên; mỗi ô đất (`1`) chưa thăm: thêm vào ngăn xếp.
- Pop một ô, đánh dấu (ở đây **sửa trực tiếp** `grid` thành `0` để vừa đánh dấu vừa tránh thăm lại), đẩy các ô đất kề.
- Cuối cùng các ô còn là `1` chính là ô bị bao vây: `sum(map(sum, grid))`.
- Dùng ngăn xếp tường minh thay vì đệ quy, vì đường đi có thể dài tới `~ 125000` ô (vượt giới hạn đệ quy mặc định của Python).

In [6]:
def enclaves_dfs(grid: List[List[int]]) -> int:
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    stack = []
    for i in range(m):
        for j in range(n):
            if (i in (0, m - 1) or j in (0, n - 1)) and g[i][j] == 1:
                g[i][j] = 0
                stack.append((i, j))
    while stack:
        i, j = stack.pop()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                g[a][b] = 0                    # đánh dấu đã thoát được (xóa khỏi tập đất)
                stack.append((a, b))
    return sum(map(sum, g))

for grid, expected in examples:
    got = enclaves_dfs(grid)
    print(f"DFS lặp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

DFS lặp = 3 OK
DFS lặp = 0 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def numEnclaves(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        stack = []
        for i in range(m):
            for j in range(n):
                if (i == 0 or i == m - 1 or j == 0 or j == n - 1) and grid[i][j] == 1:
                    grid[i][j] = 0              # đất ở biên: thoát được, xóa và loang
                    stack.append((i, j))
        while stack:
            i, j = stack.pop()
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and grid[a][b] == 1:
                    grid[a][b] = 0
                    stack.append((a, b))
        return sum(map(sum, grid))              # các ô còn lại là ô bị bao vây

sol_dfs = Solution().numEnclaves
for grid, expected in examples:
    assert sol_dfs(copy.deepcopy(grid)) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(m * n)` (mỗi ô vào ngăn xếp tối đa một lần), bộ nhớ `O(m * n)` cho ngăn xếp (tồi nhất khi cả vùng là một đường dài).

---

### 7.2. Cách 3 — BFS đa nguồn từ biên (cách nộp chính)

**Giải thích từng đoạn.**

- Đưa **mọi** ô đất trên biên vào hàng đợi `deque`, đồng thời đánh dấu đã thăm.
- Lấy một ô ra, thêm các ô đất kề **chưa thăm**.
- Kết quả: `tổng ô đất - số ô đã thăm` (đếm bằng biến `escaped` hoặc đếm ô còn là `1`).
- BFS cũng dùng `O(m * n)` bộ nhớ nhưng hàng đợi thường ngắn hơn ngăn xếp DFS trên vùng dạng đường.

In [8]:
def enclaves_bfs(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    dq = deque()
    land = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                land += 1
                if i in (0, m - 1) or j in (0, n - 1):
                    seen[i][j] = True
                    dq.append((i, j))
    escaped = len(dq)
    while dq:
        i, j = dq.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                seen[a][b] = True
                escaped += 1
                dq.append((a, b))
    return land - escaped

for grid, expected in examples:
    got = enclaves_bfs(grid)
    print(f"BFS đa nguồn = {got}", "OK" if got == expected else "SAI")
    assert got == expected

BFS đa nguồn = 3 OK
BFS đa nguồn = 0 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def numEnclaves(self, grid: List[List[int]]) -> int:
        from collections import deque
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        dq = deque()
        land = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:
                    land += 1                               # tổng số ô đất
                    if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                        seen[i][j] = True                   # đất ở biên: nguồn loang
                        dq.append((i, j))
        escaped = len(dq)
        while dq:
            i, j = dq.popleft()
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                    seen[a][b] = True
                    escaped += 1
                    dq.append((a, b))
        return land - escaped                               # ô đất không thoát được

sol_bfs = Solution().numEnclaves
for grid, expected in examples:
    assert sol_bfs(grid) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(m * n)`, bộ nhớ `O(m * n)` (ma trận `seen` và hàng đợi). Không sửa lưới đầu vào.

---

### 7.3. Cách 4 — Union-Find với nút ảo "bên ngoài"

**Ý tưởng khác.** Dùng cấu trúc **Union-Find** (hợp nhất tập hợp rời nhau). Mỗi ô `(i, j)` là một phần tử `i * n + j`; thêm một phần tử ảo `OUT = m * n` đại diện cho "bên ngoài". Với mỗi ô đất: hợp nhất nó với ô đất kề (phải và dưới), và nếu nó nằm trên biên thì hợp nhất với `OUT`. Cuối cùng, ô đất bị bao vây là ô đất **không** cùng tập với `OUT`.

**Giải thích từng đoạn.**

- `find` có **nén đường đi** (path compression) để gần như `O(1)`.
- Duyệt mỗi ô chỉ hợp nhất với ô **phải** và **dưới** (các cặp kề khác đã được xét bởi ô kia).
- Đếm ô đất có `find(ô) != find(OUT)`.

In [10]:
def enclaves_dsu(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n + 1))
    OUT = m * n

    def find(x: int) -> int:
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:                 # nén đường đi
            parent[x], x = root, parent[x]
        return root

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for i in range(m):
        for j in range(n):
            if grid[i][j] != 1:
                continue
            idx = i * n + j
            if i in (0, m - 1) or j in (0, n - 1):
                union(idx, OUT)
            if i + 1 < m and grid[i + 1][j] == 1:
                union(idx, idx + n)
            if j + 1 < n and grid[i][j + 1] == 1:
                union(idx, idx + 1)
    out_root = find(OUT)
    return sum(1 for i in range(m) for j in range(n) if grid[i][j] == 1 and find(i * n + j) != out_root)

for grid, expected in examples:
    got = enclaves_dsu(grid)
    print(f"Union-Find = {got}", "OK" if got == expected else "SAI")
    assert got == expected

Union-Find = 3 OK
Union-Find = 0 OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def numEnclaves(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        parent = list(range(m * n + 1))
        OUT = m * n                                   # nút ảo "bên ngoài"

        def find(x):
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:                  # nén đường đi
                parent[x], x = root, parent[x]
            return root

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb

        for i in range(m):
            for j in range(n):
                if grid[i][j] != 1:
                    continue
                idx = i * n + j
                if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                    union(idx, OUT)                   # đất ở biên thông với bên ngoài
                if i + 1 < m and grid[i + 1][j] == 1:
                    union(idx, idx + n)
                if j + 1 < n and grid[i][j + 1] == 1:
                    union(idx, idx + 1)
        out_root = find(OUT)
        return sum(1 for i in range(m) for j in range(n)
                   if grid[i][j] == 1 and find(i * n + j) != out_root)

sol_dsu = Solution().numEnclaves
for grid, expected in examples:
    assert sol_dsu(grid) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(m * n * alpha(m * n))` (`alpha` là hàm Ackermann ngược, gần như hằng số), bộ nhớ `O(m * n)`. Hữu ích khi cần trả lời nhiều câu hỏi về liên thông hoặc khi lưới được cập nhật dần.

## 8. TRACE từng bước

### 8.1. BFS đa nguồn cho ví dụ 1

In hàng đợi ban đầu và từng bước loang.

In [12]:
def trace_bfs(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    dq = deque()
    land = sum(map(sum, grid))
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1 and (i in (0, m - 1) or j in (0, n - 1)):
                seen[i][j] = True
                dq.append((i, j))
    print("tổng ô đất =", land, "| nguồn (đất ở biên) =", list(dq))
    escaped = len(dq)
    while dq:
        i, j = dq.popleft()
        new = []
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                seen[a][b] = True
                escaped += 1
                dq.append((a, b))
                new.append((a, b))
        print(f"lấy ({i},{j}): thêm {new or '-'}")
    print(f"thoát được {escaped} ô -> bị bao vây {land - escaped}")
    return land - escaped

assert trace_bfs(examples[0][0]) == 3
assert trace_bfs(examples[1][0]) == 0

tổng ô đất = 4 | nguồn (đất ở biên) = [(1, 0)]
lấy (1,0): thêm -
thoát được 1 ô -> bị bao vây 3
tổng ô đất = 4 | nguồn (đất ở biên) = [(0, 1), (0, 2)]
lấy (0,1): thêm -
lấy (0,2): thêm [(1, 2)]
lấy (1,2): thêm [(2, 2)]
lấy (2,2): thêm -
thoát được 4 ô -> bị bao vây 0


### 8.2. Union-Find cho ví dụ 1

In các tập sau khi hợp nhất: tập chứa `OUT` và các tập còn lại.

In [13]:
grid = examples[0][0]
m, n = len(grid), len(grid[0])
parent = list(range(m * n + 1))
OUT = m * n
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
for i in range(m):
    for j in range(n):
        if grid[i][j]:
            idx = i * n + j
            if i in (0, m - 1) or j in (0, n - 1):
                union(idx, OUT)
            if i + 1 < m and grid[i + 1][j]:
                union(idx, idx + n)
            if j + 1 < n and grid[i][j + 1]:
                union(idx, idx + 1)
groups = {}
for i in range(m):
    for j in range(n):
        if grid[i][j]:
            groups.setdefault(find(i * n + j) == find(OUT), []).append((i, j))
print("cùng tập với OUT (thoát được):", groups.get(True, []))
print("khác tập với OUT (bị bao vây):", groups.get(False, []))
assert len(groups.get(False, [])) == 3

cùng tập với OUT (thoát được): [(1, 0)]
khác tập với OUT (bị bao vây): [(1, 2), (2, 1), (2, 2)]


### 8.3. Vì sao DFS đệ quy tràn ngăn xếp

Với một "con rắn" đất dài (đường uốn khúc) nối với biên, DFS **đệ quy** đi sâu `~` số ô của con rắn. Python giới hạn độ sâu đệ quy mặc định `1000`, nên DFS đệ quy trên lưới `60 x 60` đã thất bại, huống chi `500 x 500`.

In [14]:
def snake_grid(m: int, n: int) -> List[List[int]]:
    """Con rắn đất uốn khúc ở phần bên trong, nối với biên bằng đúng một ô: DFS từ biên đi dọc cả con rắn."""
    g = [[0] * n for _ in range(m)]
    g[1][0] = 1                                         # ô đất trên biên (cột 0), nối vào đầu rắn
    for i in range(1, m - 1, 2):
        for j in range(1, n - 1):
            g[i][j] = 1                                  # hàng rắn (hàng lẻ), chỉ bên trong
        if i + 2 < m - 1:
            g[i + 1][n - 2 if (i // 2) % 2 == 0 else 1] = 1    # nối sang hàng lẻ kế tiếp, luân phiên hai đầu
    return g

def enclaves_recursive(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    def dfs(i, j):
        if not (0 <= i < m and 0 <= j < n) or g[i][j] != 1:
            return
        g[i][j] = 0
        for di, dj in DIRS:
            dfs(i + di, j + dj)
    for i in range(m):
        for j in range(n):
            if (i in (0, m - 1) or j in (0, n - 1)) and g[i][j] == 1:
                dfs(i, j)
    return sum(map(sum, g))

g = snake_grid(61, 61)
print("số ô đất của con rắn:", sum(map(sum, g)), "| đáp án đúng (Cách 3):", sol_bfs(g))
try:
    print("DFS đệ quy:", enclaves_recursive(g))
except RecursionError as e:
    print("DFS đệ quy bị RecursionError:", type(e).__name__)
assert sol_bfs(g) == 0 and sol_dfs(copy.deepcopy(g)) == 0 and enclaves_dsu(g) == 0

số ô đất của con rắn: 1800 | đáp án đúng (Cách 3): 0
DFS đệ quy bị RecursionError: RecursionError


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (BFS đa nguồn từ biên): `O(m * n)`, không đệ quy và không sửa lưới đầu vào.

```python
class Solution:
    def numEnclaves(self, grid: List[List[int]]) -> int:
        from collections import deque
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        dq = deque()
        land = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:
                    land += 1                               # tổng số ô đất
                    if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                        seen[i][j] = True                   # đất ở biên: nguồn loang
                        dq.append((i, j))
        escaped = len(dq)
        while dq:
            i, j = dq.popleft()
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                    seen[a][b] = True
                    escaped += 1
                    dq.append((a, b))
        return land - escaped                               # ô đất không thoát được
```

In [15]:
class Solution:
    def numEnclaves(self, grid: List[List[int]]) -> int:
        from collections import deque
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        dq = deque()
        land = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:
                    land += 1                               # tổng số ô đất
                    if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                        seen[i][j] = True                   # đất ở biên: nguồn loang
                        dq.append((i, j))
        escaped = len(dq)
        while dq:
            i, j = dq.popleft()
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                    seen[a][b] = True
                    escaped += 1
                    dq.append((a, b))
        return land - escaped                               # ô đất không thoát được

sol = Solution()
for grid, expected in examples:
    got = sol.numEnclaves(grid)
    print(f"{got}", "OK" if got == expected else "SAI")
    assert got == expected

3 OK
0 OK


## 10. Vì sao đúng

**Đặc trưng.** Ô đất `u` "thoát được" ⇔ tồn tại đường đi `u = c_0, c_1, ..., c_k` gồm các ô đất kề nhau (4 hướng) với `c_k` nằm trên biên. Chứng minh: ($\Leftarrow$) từ `c_k` trên biên ta bước ra khỏi lưới. ($\Rightarrow$) một dãy bước đi thoát được kết thúc bằng bước ra khỏi lưới, mà bước ra chỉ thực hiện được từ một ô trên biên, nên ô đứng ngay trước bước ra là một ô đất trên biên và các bước trước đó là các bước giữa các ô đất kề nhau.

**Cách 2 và 3.** Loang đa nguồn từ mọi ô đất trên biên thăm **đúng** tập các ô đất liên thông (qua ô đất kề nhau) với ít nhất một nguồn: tức đúng tập ô đất thoát được. Mỗi ô được thăm tối đa một lần (đánh dấu) nên `O(m * n)`. Đáp án là tổng số ô đất trừ số ô đã thăm.

**Cách 4.** Union-Find: sau khi hợp nhất mọi cặp ô đất kề nhau và mọi ô đất biên với `OUT`, hai ô đất cùng tập ⇔ liên thông qua ô đất (hoặc cùng liên thông với "bên ngoài" qua biên). Ô đất cùng tập với `OUT` ⇔ liên thông với một ô đất trên biên. Ô còn lại bị bao vây.

**Biên đặc biệt.** Nếu `m <= 2` hoặc `n <= 2` thì **mọi** ô nằm trên biên: đáp án `0`; thuật toán xử lý tự nhiên (mọi ô đất là nguồn).

**Kiểm chứng bằng máy.** Oracle (Cách 1) chạy BFS riêng cho từng ô đất theo đúng định nghĩa. Ta đối chiếu bốn cách trên **mọi** lưới có `m * n <= 16` ô (`m, n` từ `1` đến `4`) và trên nhiều lưới ngẫu nhiên có seed với nhiều mật độ đất; lưới lớn (`500 x 500`) so các cách nhanh với nhau.

In [16]:
from itertools import product
total = 0
for m in range(1, 5):
    for n in range(1, 5):
        if m * n > 16:
            continue
        for cells in product((0, 1), repeat=m * n):
            grid = [list(cells[i * n:(i + 1) * n]) for i in range(m)]
            ref = enclaves_brute(grid)
            assert enclaves_dfs(grid) == enclaves_bfs(grid) == enclaves_dsu(grid) == ref, grid
            assert sol.numEnclaves(grid) == ref
            total += 1
print("đã đối chiếu", total, "lưới (mọi lưới 1x1 .. 4x4): các cách khớp oracle")

đã đối chiếu 74954 lưới (mọi lưới 1x1 .. 4x4): các cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS riêng cho từng ô | `O((m * n)^2)` | `O(m * n)` |
| 2. DFS lặp đa nguồn | `O(m * n)` | `O(m * n)` |
| 3. BFS đa nguồn | `O(m * n)` | `O(m * n)` |
| 4. Union-Find | `O(m * n * alpha)` | `O(m * n)` |

Đo ở cỡ tối đa (`500 x 500 = 250000` ô) với các dạng lưới: ngẫu nhiên (mật độ `0.3`, `0.6`, `0.9`), toàn đất, và con rắn dài:

In [17]:
def big_snake(m: int, n: int) -> List[List[int]]:
    g = snake_grid(m, n)
    return g

rng = random.Random(11)
M = N = 500
cases = {"ngẫu nhiên p = 0.3": random_grid(rng, M, N, 0.3), "ngẫu nhiên p = 0.6": random_grid(rng, M, N, 0.6),
         "ngẫu nhiên p = 0.9": random_grid(rng, M, N, 0.9), "toàn đất": [[1] * N for _ in range(M)], "con rắn dài": big_snake(M, N)}
for label, grid in cases.items():
    ref = enclaves_bfs(grid)
    row = [f"{label:20}"]
    for name, fn in (("DFS", lambda g: sol_dfs(copy.deepcopy(g))), ("BFS", sol_bfs), ("DSU", sol_dsu), ("nộp", sol.numEnclaves)):
        t0 = time.perf_counter()
        got = fn(grid)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.1f} ms")
    print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên p = 0.3   | DFS    79.6 ms | BFS    18.5 ms | DSU    86.0 ms | nộp    19.5 ms | đáp án 73890


ngẫu nhiên p = 0.6   | DFS   165.0 ms | BFS   158.2 ms | DSU   216.4 ms | nộp   121.9 ms | đáp án 31971


ngẫu nhiên p = 0.9   | DFS   219.7 ms | BFS   235.4 ms | DSU   356.0 ms | nộp   239.5 ms | đáp án 21


toàn đất             | DFS   264.0 ms | BFS   241.8 ms | DSU   386.8 ms | nộp   261.8 ms | đáp án 0


con rắn dài          | DFS   144.2 ms | BFS   118.3 ms | DSU   135.8 ms | nộp   109.9 ms | đáp án 0


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc, cả lưới `500 x 500`) và các tính chất.

In [18]:
def all_solutions():
    return {"dfs": enclaves_dfs, "bfs": enclaves_bfs, "dsu": enclaves_dsu,
            "sol_dfs": lambda g: sol_dfs(copy.deepcopy(g)), "sol_bfs": sol_bfs, "sol_dsu": sol_dsu, "final": sol.numEnclaves}

def check_all(grid, expected=None, brute=True):
    snapshot = copy.deepcopy(grid)
    results = {name: fn(copy.deepcopy(grid)) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (len(grid), len(grid[0]), results)
    ref = results["bfs"]
    if brute and len(grid) * len(grid[0]) <= 900:
        assert enclaves_brute(grid) == ref, grid
    if expected is not None:
        assert ref == expected, (grid, ref, expected)
    assert grid == snapshot
    return ref

# 12.1 ví dụ đề
for grid, expected in examples:
    check_all(grid, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([[1]], 0),                                           # 1x1: ô đất nằm trên biên
    ([[0]], 0),                                           # không có đất
    ([[1, 0, 1]], 0),                                     # một hàng: mọi ô là biên
    ([[1], [1], [0]], 0),                                 # một cột
    ([[1, 1], [1, 1]], 0),                                # 2x2: mọi ô là biên
    ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], 1),               # một ô đất ở giữa lưới 3x3
    ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 0),               # toàn đất: ô giữa thông với biên
    ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], 0),               # toàn biển
    ([[1, 1, 1], [1, 0, 1], [1, 1, 1]], 0),               # vòng đất quanh một ô biển: không có đất bên trong
    ([[0, 0, 0, 0, 0], [0, 1, 1, 1, 0], [0, 1, 1, 1, 0], [0, 1, 1, 1, 0], [0, 0, 0, 0, 0]], 9),   # khối 3x3 bị vòng biển bao
    ([[0, 0, 0, 0, 0], [0, 1, 1, 0, 0], [0, 1, 0, 1, 1], [0, 0, 0, 1, 0]], 3),                   # tính tay ở bài tập 2: nhóm 3 ô bị bao vây
    ([[0, 0, 0, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 0, 0]], 2),                                     # hai ô đất rời nhau, đều bị bao vây
    ([[0, 1, 0], [1, 1, 1], [0, 1, 0]], 0),               # chữ thập: ô giữa thông với biên qua bốn ô biên
    ([[0, 0, 0, 0], [0, 1, 1, 0], [0, 1, 1, 0], [1, 0, 0, 0]], 4),   # chéo không tính: ô (3,0) chỉ chạm khối 2x2 theo đường chéo
]
for grid, expected in edge:
    check_all(grid, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm trên mọi lưới nhỏ, so với oracle brute)
def wrong_no_flood(grid):                         # chỉ loại các ô đất nằm TRÊN biên, không loang vào trong
    m, n = len(grid), len(grid[0])
    return sum(grid[i][j] for i in range(1, m - 1) for j in range(1, n - 1))

def wrong_partial_border(grid):                   # chỉ loang từ hàng đầu và cột đầu (quên hàng cuối, cột cuối)
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    stack = [(i, j) for i in range(m) for j in range(n) if (i == 0 or j == 0) and g[i][j] == 1]
    for i, j in stack:
        g[i][j] = 0
    while stack:
        i, j = stack.pop()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                g[a][b] = 0
                stack.append((a, b))
    return sum(map(sum, g))

def wrong_diagonal(grid):                         # cho phép đi chéo (8 hướng)
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    d8 = DIRS + ((1, 1), (1, -1), (-1, 1), (-1, -1))
    stack = [(i, j) for i in range(m) for j in range(n) if (i in (0, m - 1) or j in (0, n - 1)) and g[i][j] == 1]
    for i, j in stack:
        g[i][j] = 0
    while stack:
        i, j = stack.pop()
        for di, dj in d8:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                g[a][b] = 0
                stack.append((a, b))
    return sum(map(sum, g))

def wrong_count_regions(grid):                    # đếm số VÙNG bị bao vây thay vì số Ô
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    def flood(si, sj):
        stack = [(si, sj)]
        g[si][sj] = 0
        while stack:
            i, j = stack.pop()
            for di, dj in DIRS:
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                    g[a][b] = 0
                    stack.append((a, b))
    for i in range(m):
        for j in range(n):
            if (i in (0, m - 1) or j in (0, n - 1)) and g[i][j] == 1:
                flood(i, j)
    regions = 0
    for i in range(m):
        for j in range(n):
            if g[i][j] == 1:
                regions += 1
                flood(i, j)
    return regions

def wrong_count_escaped(grid):                    # trả về số ô đất THOÁT được (ngược yêu cầu)
    return sum(map(sum, grid)) - enclaves_bfs(grid)

def find_counter(wrong, max_cells=16):
    for m in range(1, 5):
        for n in range(1, 5):
            if m * n > max_cells:
                continue
            for cells in product((0, 1), repeat=m * n):
                grid = [list(cells[i * n:(i + 1) * n]) for i in range(m)]
                if wrong(copy.deepcopy(grid)) != enclaves_brute(grid):
                    return grid
    return None

counters = {}
for name, fn in (("chỉ loại ô đất trên biên (không loang)", wrong_no_flood), ("chỉ loang từ hàng đầu/cột đầu", wrong_partial_border),
                 ("cho phép đi chéo (8 hướng)", wrong_diagonal), ("đếm số vùng thay vì số ô", wrong_count_regions),
                 ("trả về số ô thoát được", wrong_count_escaped)):
    grid = find_counter(fn)
    assert grid is not None, name
    counters[name] = grid
    print(f"{name:40} grid={grid}: sai -> {fn(copy.deepcopy(grid))}, đúng -> {enclaves_brute(grid)}")
    check_all(grid)
print("12.3 phản ví dụ: OK")

chỉ loại ô đất trên biên (không loang)   grid=[[0, 0, 0], [0, 1, 0], [0, 1, 0]]: sai -> 1, đúng -> 0
chỉ loang từ hàng đầu/cột đầu            grid=[[0, 0], [0, 1]]: sai -> 1, đúng -> 0
cho phép đi chéo (8 hướng)               grid=[[0, 0, 0], [0, 1, 0], [0, 0, 1]]: sai -> 0, đúng -> 1
đếm số vùng thay vì số ô                 grid=[[0, 0, 0, 0], [0, 1, 1, 0], [0, 0, 0, 0]]: sai -> 1, đúng -> 2
trả về số ô thoát được                   grid=[[1]]: sai -> 1, đúng -> 0
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên: nhiều kích thước (kể cả 1 hàng / 1 cột) và mật độ đất; lưới <= 30x30 so với oracle
rng = random.Random(4020)
for _ in range(500):
    m, n = rng.randint(1, 14), rng.randint(1, 14)
    check_all(random_grid(rng, m, n, rng.choice([0.2, 0.5, 0.7, 0.9])))
for _ in range(10):
    m, n = rng.randint(15, 30), rng.randint(15, 30)
    check_all(random_grid(rng, m, n, rng.choice([0.5, 0.6, 0.7])))
print("12.4 ngẫu nhiên (510 lưới): OK")

# 12.5 lưới lớn đúng giới hạn 500 x 500: so các cách nhanh với nhau (oracle quá chậm)
rng = random.Random(5020)
for p in (0.3, 0.5, 0.6, 0.8):
    check_all(random_grid(rng, 500, 500, p), brute=False)
check_all([[1] * 500 for _ in range(500)], 0, brute=False)
check_all([[0] * 500 for _ in range(500)], 0, brute=False)
check_all(snake_grid(499, 500), 0, brute=False)
ring = [[1 if (i in (0, 499) or j in (0, 499)) else 0 for j in range(500)] for i in range(500)]
for i in range(2, 498):
    for j in range(2, 498):
        ring[i][j] = 1
check_all(ring, 496 * 496, brute=False)                    # lõi (2..497)^2 bị bao bởi một vòng biển (hàng/cột 1 và 498)
print("12.5 lưới 500 x 500: OK")

# 12.6 tính chất: đáp án <= số ô bên trong (m-2)(n-2); đối xứng khi lật/xoay/chuyển vị; thêm một ô đất thì đáp án tăng tối đa 1 hoặc giảm
rng = random.Random(6020)
for _ in range(300):
    m, n = rng.randint(1, 12), rng.randint(1, 12)
    grid = random_grid(rng, m, n, rng.choice([0.4, 0.6, 0.8]))
    ans = sol.numEnclaves(copy.deepcopy(grid))
    assert 0 <= ans <= max(0, (m - 2)) * max(0, (n - 2))
    transposed = [list(col) for col in zip(*grid)]
    flipped_ud, flipped_lr = grid[::-1], [row[::-1] for row in grid]
    for variant in (transposed, flipped_ud, flipped_lr):
        assert sol.numEnclaves(copy.deepcopy(variant)) == ans
print("12.6 tính chất (giới hạn trên, đối xứng): OK")

12.4 ngẫu nhiên (510 lưới): OK


12.5 lưới 500 x 500: OK
12.6 tính chất (giới hạn trên, đối xứng): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ loại các ô đất nằm **trên biên**, không loang vào trong

Một ô đất bên trong nhưng **kề** ô đất trên biên vẫn thoát được; chỉ bỏ ô trên biên là chưa đủ.

In [21]:
grid = counters["chỉ loại ô đất trên biên (không loang)"]
print("grid =", grid, "| sai ->", wrong_no_flood(grid), "| đúng ->", Solution().numEnclaves(copy.deepcopy(grid)))
assert wrong_no_flood(grid) != Solution().numEnclaves(copy.deepcopy(grid))
print("ví dụ 2: sai ->", wrong_no_flood(examples[1][0]), "(đúng 0)")

grid = [[0, 0, 0], [0, 1, 0], [0, 1, 0]] | sai -> 1 | đúng -> 0
ví dụ 2: sai -> 2 (đúng 0)


Vì sao sai: ở ví dụ 2 các ô `(1, 2)`, `(2, 2)` không ở biên nhưng thoát được qua `(0, 2)`. **Đúng:** loang từ mọi ô đất trên biên vào trong.

### Lỗi 2 — Chỉ loang từ một phần của biên

Biên gồm **bốn** cạnh: hàng đầu, hàng cuối, cột đầu, cột cuối. Quên hàng cuối hoặc cột cuối bỏ sót nguồn.

In [22]:
grid = counters["chỉ loang từ hàng đầu/cột đầu"]
print("grid =", grid, "| sai ->", wrong_partial_border(grid), "| đúng ->", Solution().numEnclaves(copy.deepcopy(grid)))
assert wrong_partial_border(grid) != Solution().numEnclaves(copy.deepcopy(grid))

grid = [[0, 0], [0, 1]] | sai -> 1 | đúng -> 0


Vì sao sai: đất ở hàng cuối hoặc cột cuối cũng thoát được và kéo theo cả vùng của nó. **Đúng:** `i in (0, m - 1) or j in (0, n - 1)`.

### Lỗi 3 — Cho phép đi chéo

Đề chỉ cho phép đi **4 hướng**. Đi chéo (8 hướng) nối nhầm các ô.

In [23]:
grid = counters["cho phép đi chéo (8 hướng)"]
print("grid =", grid, "| sai ->", wrong_diagonal(grid), "| đúng ->", Solution().numEnclaves(copy.deepcopy(grid)))
assert wrong_diagonal(grid) != Solution().numEnclaves(copy.deepcopy(grid))

grid = [[0, 0, 0], [0, 1, 0], [0, 0, 1]] | sai -> 0 | đúng -> 1


Vì sao sai: hai ô chéo nhau không "kề" theo đề. **Đúng:** `DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))`.

### Lỗi 4 — Đếm số vùng thay vì số ô

Đề hỏi số **ô** đất bị bao vây, không phải số vùng (đảo) bị bao vây.

In [24]:
grid = counters["đếm số vùng thay vì số ô"]
print("grid =", grid, "| sai ->", wrong_count_regions(copy.deepcopy(grid)), "| đúng ->", Solution().numEnclaves(copy.deepcopy(grid)))
assert wrong_count_regions(copy.deepcopy(grid)) != Solution().numEnclaves(copy.deepcopy(grid))

grid = [[0, 0, 0, 0], [0, 1, 1, 0], [0, 0, 0, 0]] | sai -> 1 | đúng -> 2


Vì sao sai: ví dụ 1 có một vùng ba ô bị bao vây; đáp án là `3`, không phải `1`. **Đúng:** cộng số ô.

### Lỗi 5 — Trả về số ô thoát được (ngược yêu cầu) / DFS đệ quy tràn ngăn xếp

Hai lỗi hay gặp: nhầm điều cần đếm (số ô **thoát được** thay vì **không thoát được**), và dùng DFS **đệ quy** trên lưới lớn (xem mục 8.3).

In [25]:
grid = counters["trả về số ô thoát được"]
print("grid =", grid, "| sai ->", wrong_count_escaped(grid), "| đúng ->", Solution().numEnclaves(copy.deepcopy(grid)))
assert wrong_count_escaped(grid) != Solution().numEnclaves(copy.deepcopy(grid))
g = snake_grid(61, 61)
try:
    enclaves_recursive(g)
    print("DFS đệ quy không lỗi (bất ngờ)")
except RecursionError:
    print("DFS đệ quy trên con rắn 61x61: RecursionError (cần ngăn xếp tường minh hoặc BFS)")

grid = [[1]] | sai -> 1 | đúng -> 0
DFS đệ quy trên con rắn 61x61: RecursionError (cần ngăn xếp tường minh hoặc BFS)


Vì sao sai: đề yêu cầu ô **không** thể đi ra biên. **Đúng:** `tổng ô đất - số ô thoát được`; và dùng BFS hoặc DFS lặp thay vì đệ quy.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "các ô không thể tới biên / bị bao vây" ⇒ **đảo ngược**: loang từ **biên** vào trong rồi đếm phần còn lại.
- Loang đa nguồn: đưa **mọi** nguồn vào hàng đợi/ngăn xếp trước khi chạy.
- Với lưới tới `500 x 500`: dùng **BFS hoặc DFS lặp**; DFS đệ quy dễ tràn giới hạn đệ quy mặc định.
- Union-Find với nút ảo ("bên ngoài") giải mọi bài kiểu "có nối được với biên không".
- Đánh dấu bằng cách **sửa lưới** (nhanh, ít bộ nhớ) hoặc dùng ma trận `seen` (không sửa đầu vào).
- Cùng họ: Surrounded Regions (130), Number of Closed Islands (1254), Pacific Atlantic Water Flow (417), Flood Fill (733), Number of Islands (200).
- Test nên có: lưới `1 x n`, `m x 1`, toàn `0`, toàn `1`, khối bị vòng biển bao, đất chạm nhau theo đường chéo, lưới `500 x 500` có con rắn dài.

## 15. Biến thể

1. **Surrounded Regions (130):** chữ `'O'` bị bao quanh bởi `'X'` thì lật thành `'X'`; `'O'` nối với biên giữ nguyên.
2. **Closed Islands (1254):** đếm số **đảo** (vùng liên thông) bị bao quanh hoàn toàn (ở bài đó `0` là đất, `1` là nước).
3. **Cho biết từng ô thoát được hay không** (trả về ma trận bool): chính là ma trận `seen` của Cách 3.
4. Bài liên quan: 130, 1254, 417.

Biến thể 1 (Surrounded Regions) kiểm bằng oracle chạy BFS riêng cho từng ô `'O'`:

In [26]:
def solve_surrounded(board: List[List[str]]) -> List[List[str]]:
    m, n = len(board), len(board[0])
    b = [row[:] for row in board]
    dq = deque((i, j) for i in range(m) for j in range(n) if (i in (0, m - 1) or j in (0, n - 1)) and b[i][j] == "O")
    for i, j in dq:
        b[i][j] = "S"                                  # tạm đánh dấu: nối với biên
    while dq:
        i, j = dq.popleft()
        for di, dj in DIRS:
            a, c = i + di, j + dj
            if 0 <= a < m and 0 <= c < n and b[a][c] == "O":
                b[a][c] = "S"
                dq.append((a, c))
    return [["O" if x == "S" else "X" if x == "O" else x for x in row] for row in b]

def solve_surrounded_brute(board):
    m, n = len(board), len(board[0])
    out = [row[:] for row in board]
    for si in range(m):
        for sj in range(n):
            if board[si][sj] != "O":
                continue
            seen, dq, esc = {(si, sj)}, deque([(si, sj)]), False
            while dq and not esc:
                i, j = dq.popleft()
                if i in (0, m - 1) or j in (0, n - 1):
                    esc = True
                    break
                for di, dj in DIRS:
                    a, c = i + di, j + dj
                    if board[a][c] == "O" and (a, c) not in seen:
                        seen.add((a, c))
                        dq.append((a, c))
            if not esc:
                out[si][sj] = "X"
    return out

rng = random.Random(15)
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    board = [[rng.choice("XO") for _ in range(n)] for _ in range(m)]
    assert solve_surrounded(board) == solve_surrounded_brute(board), board
b = [list("XXXX"), list("XOOX"), list("XXOX"), list("XOXX")]
print(["".join(r) for r in solve_surrounded(b)], "(ví dụ của bài 130: ô O ở hàng cuối giữ nguyên)")

['XXXX', 'XXXX', 'XXXX', 'XOXX'] (ví dụ của bài 130: ô O ở hàng cuối giữ nguyên)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS riêng từng ô | Theo định nghĩa | `O((mn)^2)` | `O(mn)` | Oracle |
| 2. DFS lặp đa nguồn | Loang từ biên, xóa ô đã thoát | `O(mn)` | `O(mn)` | Ít bộ nhớ phụ (sửa lưới) |
| 3. BFS đa nguồn | Loang từ biên, ma trận `seen` | `O(mn)` | `O(mn)` | **Cách nộp chính** (không sửa lưới) |
| 4. Union-Find | Nút ảo "bên ngoài" | `O(mn alpha)` | `O(mn)` | Khi cần truy vấn liên thông |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
g = [[0, 1, 0], [1, 1, 1], [0, 1, 0]]
m, n = len(g), len(g[0])
print([(i, j) for i in range(m) for j in range(n) if i in (0, m - 1) or j in (0, n - 1)].__len__())
print(sum(map(sum, g)), Solution().numEnclaves(g))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # số ô trên biên
my_guess_2 = None      # hai số

### Bài 2 — Tính tay (dễ-vừa)

Lưới `4 x 5`:

```
0 0 0 0 0
0 1 1 0 0
0 1 0 1 1
0 0 0 1 0
```

Tìm số ô đất bị bao vây.

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS đa nguồn.

In [29]:
def exercise_3(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    dq = deque()
    land = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                land += 1
                # TODO 1: nếu (i, j) nằm trên biên: đánh dấu seen và đưa vào dq
    escaped = len(dq)
    while dq:
        i, j = dq.popleft()
        for di, dj in DIRS:
            # TODO 2: tính ô kề (a, b); nếu trong lưới, là đất, chưa thăm thì đánh dấu, tăng escaped, thêm vào dq
            pass
    # TODO 3: trả về land - escaped
    pass


# for grid, expected in examples:
#     assert exercise_3(grid) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(grid)` bằng **Union-Find** với nút ảo "bên ngoài".

In [30]:
def exercise_4(grid: List[List[int]]) -> int:
    # TODO
    pass


# for grid, expected in examples:
#     assert exercise_4(grid) == expected

### Bài 5 — Mini challenge: đếm số vùng bị bao vây (khó)

Viết `exercise_5(grid)` trả về **số vùng đất liên thông** (4 hướng) bị bao vây hoàn toàn (không có ô nào thoát được ra biên).

```python
exercise_5([[0,0,0,0],[1,0,1,0],[0,1,1,0],[0,0,0,0]])        -> 1     # vùng {(1,2),(2,1),(2,2)}
exercise_5([[0,1,1,0],[0,0,1,0],[0,0,1,0],[0,0,0,0]])        -> 0
exercise_5([[0,0,0,0,0],[0,1,0,1,0],[0,0,0,0,0]])            -> 2
exercise_5([[0,0,0,0,0],[0,1,1,0,0],[0,1,0,1,1],[0,0,0,1,0]]) -> 1
```
Gợi ý: loang từ biên và xóa; rồi đếm số lần phải loang mới trên các ô còn lại.

In [31]:
def exercise_5(grid: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5([[0,0,0,0],[1,0,1,0],[0,1,1,0],[0,0,0,0]]) == 1
# assert exercise_5([[0,1,1,0],[0,0,1,0],[0,0,1,0],[0,0,0,0]]) == 0
# assert exercise_5([[0,0,0,0,0],[0,1,0,1,0],[0,0,0,0,0]]) == 2
# assert exercise_5([[0,0,0,0,0],[0,1,1,0,0],[0,1,0,1,1],[0,0,0,1,0]]) == 1

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: lưới `3 x 3` có `9 - 1 = 8` ô trên biên (chỉ ô giữa không ở biên).
- Mức 2: có `5` ô đất (chữ thập); ô giữa `(1, 1)` liên thông với `(0, 1)` trên biên.
- Mức 3: dòng 1: `8`; dòng 2: `5 0`.

**Bài 2**
- Mức 1: đất trên biên: `(2, 4)` (cột cuối). Loang: `(2, 3)` kề `(2, 4)` rồi `(3, 3)` kề `(2, 3)`.
- Mức 2: các ô còn lại: `(1, 1)`, `(1, 2)`, `(2, 1)`.
- Mức 3: `3`.

**Bài 3**
- Mức 1: `if i in (0, m - 1) or j in (0, n - 1): seen[i][j] = True; dq.append((i, j))`.
- Mức 2: `a, b = i + di, j + dj`; `if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:`.
- Mức 3: `seen[a][b] = True; escaped += 1; dq.append((a, b))`; cuối `return land - escaped`.

**Bài 4**
- Mức 1: `parent = list(range(m * n + 1))`, `OUT = m * n`.
- Mức 2: với mỗi ô đất: hợp nhất với `OUT` nếu ở biên; hợp nhất với ô phải và ô dưới nếu là đất.
- Mức 3: đếm ô đất có `find(ô) != find(OUT)`.

**Bài 5**
- Mức 1: sao chép lưới; loang từ biên xóa các ô thoát được.
- Mức 2: duyệt các ô còn lại; gặp ô đất thì `regions += 1` và loang xóa cả vùng.
- Mức 3: trả về `regions`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
g = [[0, 1, 0], [1, 1, 1], [0, 1, 0]]
m, n = len(g), len(g[0])
print([(i, j) for i in range(m) for j in range(n) if i in (0, m - 1) or j in (0, n - 1)].__len__())
print(sum(map(sum, g)), Solution().numEnclaves(copy.deepcopy(g)))

assert len([(i, j) for i in range(m) for j in range(n) if i in (0, m - 1) or j in (0, n - 1)]) == 8
assert (sum(map(sum, g)), Solution().numEnclaves(copy.deepcopy(g))) == (5, 0)

8
5 0


### Đáp án Bài 2

Đất trên biên: `(2, 4)`; loang: `(2, 3)`, rồi `(3, 3)`. Còn lại `(1, 1)`, `(1, 2)`, `(2, 1)`: `3` ô bị bao vây.

In [33]:
hand_answer_solution = 3
g = [[0, 0, 0, 0, 0], [0, 1, 1, 0, 0], [0, 1, 0, 1, 1], [0, 0, 0, 1, 0]]
assert Solution().numEnclaves(copy.deepcopy(g)) == hand_answer_solution == enclaves_brute(g)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_solution(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    dq = deque()
    land = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1:
                land += 1
                if i in (0, m - 1) or j in (0, n - 1):
                    seen[i][j] = True
                    dq.append((i, j))
    escaped = len(dq)
    while dq:
        i, j = dq.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                seen[a][b] = True
                escaped += 1
                dq.append((a, b))
    return land - escaped

for grid, expected in examples:
    assert exercise_3_solution(grid) == expected
rng = random.Random(3)
for _ in range(300):
    g = random_grid(rng, rng.randint(1, 10), rng.randint(1, 10), 0.6)
    assert exercise_3_solution(g) == enclaves_brute(g)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_solution(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n + 1))
    OUT = m * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for i in range(m):
        for j in range(n):
            if grid[i][j] != 1:
                continue
            idx = i * n + j
            if i in (0, m - 1) or j in (0, n - 1):
                union(idx, OUT)
            if i + 1 < m and grid[i + 1][j] == 1:
                union(idx, idx + n)
            if j + 1 < n and grid[i][j + 1] == 1:
                union(idx, idx + 1)
    root = find(OUT)
    return sum(1 for i in range(m) for j in range(n) if grid[i][j] == 1 and find(i * n + j) != root)

for grid, expected in examples:
    assert exercise_4_solution(grid) == expected
rng = random.Random(4)
for _ in range(300):
    g = random_grid(rng, rng.randint(1, 10), rng.randint(1, 10), 0.6)
    assert exercise_4_solution(g) == enclaves_brute(g)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_solution(grid: List[List[int]]) -> int:
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])

    def flood(si, sj):
        stack = [(si, sj)]
        g[si][sj] = 0
        while stack:
            i, j = stack.pop()
            for di, dj in DIRS:
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                    g[a][b] = 0
                    stack.append((a, b))

    for i in range(m):
        for j in range(n):
            if (i in (0, m - 1) or j in (0, n - 1)) and g[i][j] == 1:
                flood(i, j)
    regions = 0
    for i in range(m):
        for j in range(n):
            if g[i][j] == 1:
                regions += 1
                flood(i, j)
    return regions

assert exercise_5_solution([[0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]]) == 1
assert exercise_5_solution([[0, 1, 1, 0], [0, 0, 1, 0], [0, 0, 1, 0], [0, 0, 0, 0]]) == 0
assert exercise_5_solution([[0, 0, 0, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 0, 0]]) == 2
assert exercise_5_solution([[0, 0, 0, 0, 0], [0, 1, 1, 0, 0], [0, 1, 0, 1, 1], [0, 0, 0, 1, 0]]) == 1
rng = random.Random(5)
for _ in range(300):
    g = random_grid(rng, rng.randint(1, 10), rng.randint(1, 10), 0.5)
    assert exercise_5_solution(g) == wrong_count_regions(g)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (BFS đa nguồn từ biên)

```python
class Solution:
    def numEnclaves(self, grid: List[List[int]]) -> int:
        from collections import deque
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        dq = deque()
        land = 0
        for i in range(m):
            for j in range(n):
                if grid[i][j] == 1:
                    land += 1
                    if i == 0 or i == m - 1 or j == 0 or j == n - 1:
                        seen[i][j] = True                   # đất ở biên: nguồn loang
                        dq.append((i, j))
        escaped = len(dq)
        while dq:
            i, j = dq.popleft()
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                    seen[a][b] = True
                    escaped += 1
                    dq.append((a, b))
        return land - escaped
```

Công thức nhớ nhanh: đáp án `= (tổng ô đất) - (số ô đất liên thông với một ô đất trên biên)`; loang từ **cả bốn cạnh** của biên, 4 hướng.

### Checklist trước khi submit

- [ ] Nguồn loang là **mọi** ô đất trên biên (hàng đầu, hàng cuối, cột đầu, cột cuối)?
- [ ] Chỉ đi **4 hướng** (không chéo), kiểm tra chỉ số trong lưới?
- [ ] Đánh dấu đã thăm để mỗi ô vào hàng đợi/ngăn xếp tối đa một lần?
- [ ] Dùng BFS hoặc DFS **lặp** (không đệ quy) cho lưới `500 x 500`?
- [ ] Đếm số **ô** không thoát được (không phải số vùng, không phải số ô thoát được)?
- [ ] Đã thử: `1 x n`, `m x 1`, toàn `0`, toàn `1`, khối bị vòng biển bao, đất chạm nhau theo đường chéo?